In [8]:
import sqlite3

In [9]:
import sqlite3

db_path = "../database/logistics.db"

con = sqlite3.connect(db_path)

In [10]:
con.execute("SELECT 1 AS test").fetchall()

[(1,)]

## Load the Olist tables

In [11]:
import pandas as pd

data_path = "../data/raw/"

orders = pd.read_csv(data_path + "olist_orders_dataset.csv")
customers = pd.read_csv(data_path + "olist_customers_dataset.csv")
order_items = pd.read_csv(data_path + "olist_order_items_dataset.csv")
products = pd.read_csv(data_path + "olist_products_dataset.csv")
sellers = pd.read_csv(data_path + "olist_sellers_dataset.csv")
payments = pd.read_csv(data_path + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(data_path + "olist_order_reviews_dataset.csv")

In [12]:
orders.to_sql("orders", con, if_exists="replace", index=False)
customers.to_sql("customers", con, if_exists="replace", index=False)
order_items.to_sql("order_items", con, if_exists="replace", index=False)
products.to_sql("products", con, if_exists="replace", index=False)
sellers.to_sql("sellers", con, if_exists="replace", index=False)
payments.to_sql("payments", con, if_exists="replace", index=False)
reviews.to_sql("reviews", con, if_exists="replace", index=False)

99224

In [13]:
tables = con.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
""").fetchall()

tables

[('customers',),
 ('order_items',),
 ('orders',),
 ('payments',),
 ('products',),
 ('reviews',),
 ('sellers',)]

### Run our first real SQL query

In [14]:
query = """
SELECT
    COUNT(*) AS total_orders
FROM orders;
"""

result = con.execute(query).fetchall()

result

[(99441,)]

###  Late Delivery Rat

In [18]:
query = """
SELECT
    COUNT(*) AS total_delivered_orders,

    SUM(
        CASE
            WHEN order_delivered_customer_date > order_estimated_delivery_date
            THEN 1
            ELSE 0
        END
    ) AS late_orders,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN order_delivered_customer_date > order_estimated_delivery_date
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS late_delivery_rate

FROM orders

WHERE order_delivered_customer_date IS NOT NULL;
"""

result = con.execute(query).fetchall()

result

[(96476, 7827, 8.11)]

### Create a logistics analytics view

In [19]:
query = """
CREATE VIEW IF NOT EXISTS delivery_performance AS

SELECT
    order_id,
    customer_id,
    order_purchase_timestamp,
    order_delivered_customer_date,
    order_estimated_delivery_date,

    CASE
        WHEN order_delivered_customer_date IS NOT NULL
        THEN CAST(
            julianday(order_delivered_customer_date)
            - julianday(order_estimated_delivery_date)
            AS INTEGER
        )
    END AS delay_days,

    CASE
        WHEN order_delivered_customer_date IS NULL
            THEN 'Not Delivered'
        WHEN order_delivered_customer_date > order_estimated_delivery_date
            THEN 'Late'
        ELSE 'On Time'
    END AS delivery_status

FROM orders;
"""

con.execute(query)

In [20]:
con.execute("""
    SELECT
        delivery_status,
        COUNT(*) AS orders
    FROM delivery_performance
    GROUP BY delivery_status;
""").fetchall()

[('Late', 7827), ('Not Delivered', 2965), ('On Time', 88649)]

### schema/business context

In [21]:
schema_description = """
You are a logistics data analyst.

You have access to the following SQLite view:

delivery_performance

Columns:
- order_id: unique order identifier
- customer_id: customer identifier
- order_purchase_timestamp: timestamp when the order was placed
- order_delivered_customer_date: actual customer delivery timestamp
- order_estimated_delivery_date: estimated delivery date
- delay_days: number of days between actual and estimated delivery
- delivery_status: Late, On Time, or Not Delivered

Business definitions:
- A delivered order has a non-null order_delivered_customer_date.
- A late order has delay_days > 0.
- Late delivery rate = late delivered orders / total delivered orders * 100.
"""

### Load API key

In [22]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

api_key = os.getenv("OPENAI_API_KEY")

print("API key loaded:", api_key is not None)

API key loaded: True


In [23]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

print("OpenAI client created successfully.")

OpenAI client created successfully.


In [25]:
import ollama

response = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "user",
            "content": "Explain late delivery in one sentence."
        }
    ]
)

print(response["message"]["content"])

Late delivery refers to the failure to meet a scheduled or agreed-upon delivery date or time, resulting in the non-arrival of goods or services within the expected timeframe.


### database context

In [26]:
schema_description = """
You are a logistics data analyst.

You have access to the following SQLite view:

delivery_performance

Columns:
- order_id: unique order identifier
- customer_id: customer identifier
- order_purchase_timestamp: timestamp when the order was placed
- order_delivered_customer_date: actual customer delivery timestamp
- order_estimated_delivery_date: estimated delivery date
- delay_days: number of days between actual and estimated delivery
- delivery_status: Late, On Time, or Not Delivered

Business definitions:
- A delivered order has a non-null order_delivered_customer_date.
- A late order has delay_days > 0.
- Late delivery rate = late delivered orders / total delivered orders * 100.
"""

In [27]:
user_question = "What is the late delivery rate?"

print(user_question)

What is the late delivery rate?


### Text-to-SQL

In [28]:
prompt = f"""
{schema_description}

User question:
{user_question}

Write a SQLite SQL query that answers the user's question.

Return only the SQL query.
Do not include explanations.
"""

response = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

generated_sql = response["message"]["content"]

print(generated_sql)

SELECT (SUM(CASE WHEN delay_days > 0 THEN 1 ELSE 0 END) / COUNT(*)) * 100 FROM delivery_performance


In [48]:
prompt = f"""
{schema_description}

User question:
{user_question}

Instructions:
- Translate the user's question into valid SQLite SQL.
- Use only the delivery_performance view.
- Use the column definitions and business definitions provided above.
- Include only delivered orders when the question concerns delivered-order performance.
- For late orders, use delay_days > 0.
- For percentage calculations, use 100.0 to avoid SQLite integer division.
- Do not assume the user is asking for late_delivery_rate unless the question explicitly asks for a rate or percentage.
- Return only SQL. No explanation.
"""

response = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

generated_sql = response["message"]["content"]

print(generated_sql)

SELECT (100.0 * COUNT(CASE WHEN delivery_status = 'Late' AND order_delivered_customer_date IS NOT NULL THEN 1 END) 
        / (SELECT COUNT(order_id) FROM delivery_performance WHERE order_delivered_customer_date IS NOT NULL)) * 100.0


In [30]:
prompt = f"""
{schema_description}

User question:
{user_question}

For this specific metric, use this exact calculation pattern:

late_delivery_rate =
    COUNT of delivered orders where delay_days > 0
    divided by
    COUNT of all delivered orders
    multiplied by 100

Equivalent SQL logic:

SUM(CASE WHEN delay_days > 0 THEN 1 ELSE 0 END)
/
COUNT(*)

The query must:
- use only delivery_performance
- include only delivered orders
- calculate late orders divided by delivered orders
- return the result as late_delivery_rate
- use SQLite syntax

Return only the SQL query.
"""

response = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

generated_sql = response["message"]["content"]

print(generated_sql)

SELECT (SUM(CASE WHEN delay_days > 0 THEN 1 ELSE 0 END) / COUNT(*)) * 100 FROM delivery_performance


In [40]:
prompt = f"""
{schema_description}

User question:
{user_question}

For this specific metric, use this exact calculation:

late_delivery_rate =
late delivered orders / total delivered orders * 100

Use this SQL logic:

SUM(CASE WHEN delay_days > 0 THEN 1 ELSE 0 END)
/
COUNT(*)

Requirements:
- Use only delivery_performance.
- Include only delivered orders.
- Late orders have delay_days > 0.
- Calculate late orders divided by delivered orders.
- Return the column as late_delivery_rate.
- For percentage calculations, use 100.0 instead of 100 to avoid SQLite integer division.
- Use valid SQLite SQL.
- Return only SQL. No explanation.

"""

response = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

generated_sql = response["message"]["content"]

print(generated_sql)

SELECT 100.0 * SUM(CASE WHEN delay_days > 0 THEN 1 ELSE 0 END) / COUNT(*) FROM delivery_performance WHERE order_delivered_customer_date IS NOT NULL


### Create a validator

In [34]:
def validate_sql(sql):
    sql_clean = sql.strip().lower()

    # Must be a SELECT statement
    if not sql_clean.startswith("select"):
        return False, "Only SELECT queries are allowed."

    # Block potentially dangerous SQL commands
    blocked_keywords = [
        "insert",
        "update",
        "delete",
        "drop",
        "alter",
        "create",
        "replace",
        "attach",
        "detach"
    ]

    for keyword in blocked_keywords:
        if keyword in sql_clean:
            return False, f"Blocked SQL keyword: {keyword}"

    # Only allow our approved view
    if "delivery_performance" not in sql_clean:
        return False, "Query must use the delivery_performance view."

    return True, "SQL passed validation."

In [35]:
is_valid, message = validate_sql(generated_sql)

print(is_valid)
print(message)

True
SQL passed validation.


In [36]:
if is_valid:
    result = con.execute(generated_sql).fetchall()
    print(result)
else:
    print(message)

[(0,)]


In [37]:
result_text = str(result)

answer_prompt = f"""
You are a logistics data analyst.

User question:
{user_question}

SQL result:
{result_text}

Answer the user's question using only the SQL result.

Requirements:
- Give the answer clearly.
- Include the percentage if the result is a percentage.
- Do not invent any additional numbers.
- Keep the answer concise and business-friendly.
"""

response = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "user",
            "content": answer_prompt
        }
    ]
)

final_answer = response["message"]["content"]

print(final_answer)

There is no data on late deliveries.


In [38]:
print("SQL:")
print(generated_sql)

print("\nResult:")
print(result)

SQL:
SELECT (SUM(CASE WHEN delay_days > 0 THEN 1 ELSE 0 END) / COUNT(*)) * 100 FROM delivery_performance WHERE order_delivered_customer_date IS NOT NULL

Result:
[(0,)]


In [39]:
corrected_sql = """
SELECT
    ROUND(
        100.0 * SUM(CASE WHEN delay_days > 0 THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS late_delivery_rate
FROM delivery_performance
WHERE order_delivered_customer_date IS NOT NULL;
"""

corrected_result = con.execute(corrected_sql).fetchall()

print(corrected_result)

[(6.77,)]


In [41]:
is_valid, message = validate_sql(generated_sql)

print("Validation:", is_valid)
print("Message:", message)

if is_valid:
    result = con.execute(generated_sql).fetchall()
    print("Result:", result)
else:
    print("SQL was blocked.")

Validation: True
Message: SQL passed validation.
Result: [(6.773705377503213,)]


In [53]:
def ask_logistics_question(user_question):
    
    # 1. Generate SQL
    prompt = f"""
    You are a logistics data analyst responsible for translating business questions into SQL.

    Database schema and business definitions:
    {schema_description}

    User question:
    {user_question}

    Instructions:
    - Translate the user's question into valid SQLite SQL.
    - Use only the approved delivery_performance view.
    - Use the column definitions and business definitions provided above.
    - When the question concerns delivered orders, exclude orders where order_delivered_customer_date IS NULL.
    - A late order is defined as delay_days > 0.
    - An on-time order is defined as delivery_status = 'On Time'.
    - A delivered order is defined as delivery_status IN ('Late', 'On Time').
    - For percentage calculations, use 100.0 instead of 100 to avoid SQLite integer division.
    - Do not assume the user is asking for a percentage unless the question asks for a rate, percentage, or proportion.
    - Do not assume the user is asking for late deliveries unless the question refers to late or delayed deliveries.
    - Select only the columns needed to answer the user's question.
    - Do not invent columns, tables, metrics, or business definitions.
    - Return only the SQL query.
    - Do not include markdown code fences.
    - Do not include explanations.
    """

    response = ollama.chat(
        model="llama3.2",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    generated_sql = response["message"]["content"].strip()

    # 2. Validate SQL
    is_valid, message = validate_sql(generated_sql)

    if not is_valid:
        return f"SQL validation failed: {message}"

    # 3. Execute SQL
    try:
        result = con.execute(generated_sql).fetchall()
    except Exception as e:
        return f"SQL execution failed: {e}"

    # 4. Generate business answer
    answer_prompt = f"""
    You are a logistics data analyst.

    User question:
    {user_question}

    SQL result:
    {result}

    Answer the user's question using only the SQL result.

    Requirements:
    - State the result clearly.
    - Do not invent numbers.
    - Do not claim there is no data unless the SQL result actually indicates no data.
    - Keep the answer concise and business-friendly.
    """

    response = ollama.chat(
        model="llama3.2",
        messages=[
            {
                "role": "user",
                "content": answer_prompt
            }
        ]
    )

    final_answer = response["message"]["content"].strip()

    return {
        "question": user_question,
        "sql": generated_sql,
        "result": result,
        "answer": final_answer
    }

In [44]:
answer = ask_logistics_question(
    "How many orders were delivered late?"
)

print(answer)

100 orders were delivered late.


In [49]:
answer = ask_logistics_question(
    "How many orders were delivered late?"
)

print(answer)

According to the data, 100 orders were delivered late.


In [55]:
response = ask_logistics_question(
    "How many orders were delivered late?"
)

print("Question:", response["question"])
print("\nSQL:", response["sql"])
print("\nResult:", response["result"])
print("\nAnswer:", response["answer"])

Question: How many orders were delivered late?

SQL: SELECT COUNT(order_id) FROM delivery_performance WHERE delivery_status = 'Late'

Result: [(7827,)]

Answer: There were 7,827 orders delivered late.


In [56]:
response = ask_logistics_question(
    "How many orders were not delivered?"
)

print("Question:", response["question"])
print("\nSQL:", response["sql"])
print("\nResult:", response["result"])
print("\nAnswer:", response["answer"])

Question: How many orders were not delivered?

SQL: SELECT COUNT(order_id) FROM delivery_performance WHERE delivery_status = 'Not Delivered'

Result: [(2965,)]

Answer: There were 2965 orders that were not delivered.
